# Cursul 4. Curățarea și integrarea datelor
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

**Curățarea înseamnă verificarea datelor înainte de folosire.** În C3 am adus sursele la aceeași formă (preprocesarea); azi verificăm dacă valorile sunt corecte, complete și comparabile, apoi unim tabelele. Pașii urmează etapele clasice ale curățării:

| Etapa | Întrebarea | În pandas | Secțiunea |
|---|---|---|---|
| Profilarea | Ce conține tabelul? | `info`, `value_counts`, `describe` | 2 |
| Tipurile | Numerele sunt numere, codurile sunt text? | `dtypes`, `pd.to_numeric` | 3 |
| Duplicatele | Apare aceeași cheie de două ori? | `duplicated`, `drop_duplicates` | 4 |
| Valorile imposibile | Există procente peste 100, valori negative, valori neobișnuite? | `describe`, `nsmallest`, un filtru | 5 |
| Lipsurile | Ce înseamnă „lipsă”, câte valori lipsesc și unde? | `isna`, `count` | 6 |
| Codurile | Aceeași țară are același cod în toate sursele? | `replace` | 7 |
| Transformarea și integrarea | Cum ajungem la un tabel de analiză? | `pivot`, `merge` | 8-11 |
| Validarea | Tabelul final respectă regulile lui? | o celulă de verificări | 12 |

La integrare se pierd date fără nicio eroare, de aceea o singură regulă: **după fiecare `merge`, numărăm rândurile.**

Tabelele finale merg tot în `data/processed`. În C5 le încărcăm într-o bază de date.

In [ ]:
import pandas as pd

## 1. Deschidem tabelele din C3
Ca în C3, le citim din depozitul cursului: sunt exact tabelele salvate de caietul 3, deci toți lucrăm pe aceleași date. Adresa folderului e aceeași, se schimbă doar numele fișierului.

In [ ]:
DATE_C3 = "https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/processed/"   # tabelele din C3

indicatori = pd.read_csv(DATE_C3 + "indicatori.csv")      # șapte indicatori, pe regiuni NUTS2
nume = pd.read_csv(DATE_C3 + "regiuni.csv")               # nomenclatorul: codul, țara, numele, nivelul
sci = pd.read_csv(DATE_C3 + "sci_regiuni.csv")            # SCI pe perechi de regiuni

## 2. Profilarea: ce conține tabelul
Înainte de orice decizie, descriem tabelul: câte rânduri și coloane are, de ce tip e fiecare coloană și câte valori are.

In [ ]:
indicatori.info()

**Interpretează:** 2.281 de rânduri, dar coloana `valoare` are doar 1.893 de valori (*Non-Null Count*). Diferența, 388, sunt valori lipsă. `info()` arată dintr-o privire și tipul fiecărei coloane.

`value_counts()` numără de câte ori apare fiecare valoare dintr-o coloană. Pe `indicator`, arată câte regiuni are fiecare indicator.

In [ ]:
indicatori["indicator"].value_counts()

Indicatorii nu acoperă același număr de regiuni: de la 354 (vârsta medie) la 226 (rețelele sociale). E primul semn că lipsurile nu sunt distribuite uniform.

O verificare rapidă a codurilor: lungimea codului spune nivelul (C3). Toate ar trebui să aibă 4 caractere, adică NUTS2.

In [ ]:
indicatori["regiune"].str.len().value_counts()     # lungimea codurilor, numărată

**Exercițiu.** Nomenclatorul `nume` conține toate nivelurile. Câte coduri are fiecare nivel?

In [ ]:
# Câte coduri are fiecare nivel: value_counts pe coloana nivel, apoi sort_index()

## 3. Tipurile
**Anticipează:** ce tip are coloana `valoare`? Dar `regiune`?

In [ ]:
indicatori.dtypes

`valoare` este `float64`, număr cu zecimale; o valoare lipsă apare ca `NaN`. `regiune` și `indicator` sunt `object`, adică text.

Dacă printre numere apare un text (Eurostat scrie uneori `:` în loc de o valoare), toată coloana devine `object` și calculele nu mai merg. `pd.to_numeric` transformă textul în numere, iar cu `errors="coerce"` ce nu e număr devine valoare lipsă:

In [ ]:
exemplu = pd.Series(["12.5", ":", "7"])           # o coloană citită ca text
pd.to_numeric(exemplu, errors="coerce")            # ":" devine NaN, restul devin numere

## 4. Duplicatele
Un rând din `indicatori` e identificat de **regiune + indicator**: aceasta e cheia tabelului. Nicio cheie nu trebuie să apară de două ori. `duplicated` marchează rândurile care repetă o cheie deja întâlnită.

In [ ]:
# Câte rânduri repetă cheia regiune + indicator: duplicated cu lista celor două coloane, apoi .sum()

0: fiecare cheie apare o singură dată. În SCI, cheia e perechea de regiuni:

In [ ]:
print(len(sci), "rânduri;", sci.duplicated(["regiune_1", "regiune_2"]).sum(), "chei repetate")
doua = ["RO11", "ITC4"]                                            # Nord-Vest și Lombardia
sci[sci["regiune_1"].isin(doua) & sci["regiune_2"].isin(doua) & (sci["regiune_1"] != sci["regiune_2"])]

Nicio cheie repetată, totuși aceeași legătură apare de două ori: RO11-ITC4 și ITC4-RO11, cu același SCI. Este o dublură de **conținut**, nu de cheie, și `duplicated` nu o vede. O rezolvăm la secțiunea 9.

`drop_duplicates` păstrează prima apariție a fiecărei valori. Așa obținem lista celor 298 de regiuni din SCI, aria proiectului:

In [ ]:
regiuni_sci = sci["regiune_1"].drop_duplicates()     # fiecare regiune, o singură dată
len(regiuni_sci)

## 5. Valorile imposibile sau neobișnuite
Un rezumat pe indicator: minimul, mediana, maximul.

In [ ]:
indicatori.groupby("indicator")["valoare"].describe().round(1)

**Interpretează:** populația minimă este 0. Care regiune e?

In [ ]:
populatie = indicatori[indicatori["indicator"] == "populatie"]
populatie.nsmallest(3, "valoare")          # cele mai mici trei valori

`NO0B` este Svalbard, arhipelagul norvegian: Eurostat raportează chiar 0. Nu e o eroare. **Valorile neobișnuite nu se șterg automat**: întâi se verifică în sursă.

**Exercițiu.** Studiile superioare, șomajul și rețelele sociale sunt procente, deci între 0 și 100. Câte valori sunt peste 100?

In [ ]:
# 1. Rândurile cu indicatorii în procente: studii_superioare, somaj, retele_sociale (cu isin)
# 2. Câte valori sunt peste 100? (o condiție, apoi .sum())

## 6. Lipsurile
### 6.1. Ce înseamnă „lipsă”
Pentru pandas, o valoare lipsă e `NaN`. La citire, pandas transformă automat în `NaN` câteva texte: celula goală, dar și `NA`, `N/A`, `null`. La noi, `NA` este codul Namibiei. Comparăm cele două citiri ale migrației:

In [ ]:
gresit = pd.read_csv(DATE_C3 + "migratie.csv")                                           # citirea implicită
migratie = pd.read_csv(DATE_C3 + "migratie.csv", keep_default_na=False, na_values=[""])  # doar celula goală e lipsă

print("cetățenii lipsă, citirea implicită:", gresit["cetatenie"].isna().sum())
print("cetățenii lipsă, citirea corectă:  ", migratie["cetatenie"].isna().sum())

44 de rânduri cu cetățenii namibieni ar fi devenit „lipsă”, fără niciun mesaj. De aceea codurile de țară se citesc mereu cu `keep_default_na=False`.

`value_counts()` ascunde implicit valorile lipsă. Cu `dropna=False`, le numără și pe ele:

In [ ]:
zboruri = pd.read_csv(DATE_C3 + "zboruri.csv", keep_default_na=False, na_values=[""])
# Câte perechi au și câte nu au pasageri: isna() pe coloana pasageri, apoi value_counts(dropna=False)

### 6.2. Câte valori lipsesc și unde
Numărăm lipsurile doar pentru regiunile din SCI, aria proiectului. `size` numără rândurile, `count` doar valorile prezente; diferența sunt lipsurile.

In [ ]:
in_sci = indicatori[indicatori["regiune"].isin(regiuni_sci)]           # doar regiunile din SCI
in_sci.groupby("indicator")["valoare"].agg(["size", "count"])

**Interpretează:** rețelele sociale au valori pentru 174 din 210 rânduri, iar unele regiuni nu au rând deloc pentru unii indicatori. Ce facem cu lipsurile decidem după ce vedem tabelul regiunilor (secțiunea 8).

## 7. Codurile
O capcană care nu dă nicio eroare: Eurostat scrie Grecia `EL` și Regatul Unit `UK`, iar SCI le scrie `GR` și `GB`. Dacă nu le aliniem, aceste țări se pierd la lipire, fără niciun mesaj. `replace` înlocuiește valorile după un dicționar.

In [ ]:
coduri = {"EL": "GR", "UK": "GB"}                    # codul Eurostat -> codul SCI

migratie = migratie.replace({"tara_rezidenta": coduri, "cetatenie": coduri})    # în ambele coloane
zboruri = zboruri.replace({"tara_1": coduri, "tara_2": coduri})

migratie["tara_rezidenta"].isin(["EL", "UK"]).sum()     # verificare: nu mai rămâne niciun EL sau UK

## 8. Tabelul regiunilor: din lung în lat
Trecem din formatul **lung** (un rând pe regiune și indicator) în formatul **lat** (un rând pe regiune, o coloană pe indicator). `pivot` face asta.

In [ ]:
valori = in_sci.pivot(index="regiune", columns="indicator", values="valoare")   # rânduri, coloane, celule
valori = valori.reset_index()                                                   # regiunea, din nou coloană
valori.head()

Pornim de la lista regiunilor din SCI și lipim numele și indicatorii. `how="left"` păstrează toate regiunile din SCI, chiar și pe cele fără date. Apoi regula integrării: numărăm rândurile.

In [ ]:
regiuni = pd.DataFrame({"regiune": regiuni_sci})                                     # cele 298 de regiuni din SCI
regiuni = regiuni.merge(nume[["regiune", "tara", "denumire"]], on="regiune", how="left")
regiuni = regiuni.merge(valori, on="regiune", how="left")

print(len(regiuni_sci), "->", len(regiuni), "regiuni")     # trebuie să fie același număr

Același număr: fiecare regiune a primit cel mult un rând. Dacă numărul ar crește, o regiune ar fi apărut de două ori în tabelul lipit; dacă ar scădea, am fi pierdut regiuni.

Câte valori lipsesc, pe coloană, și ce procent reprezintă:

In [ ]:
lipsuri = pd.DataFrame({"lipsa": regiuni.isna().sum(),                       # câte
                        "procent": (regiuni.isna().mean() * 100).round(1)})  # ce procent din regiuni
lipsuri

`isna()` dă `True` unde lipsește o valoare; `sum()` numără valorile `True`, iar `mean()` dă proporția lor. Rețelele sociale lipsesc pentru 42% dintre regiuni, restul indicatorilor pentru sub 8%.

Dacă am păstra doar regiunile complete, am pierde multe regiuni, iar pierderea nu e întâmplătoare: lipsurile sunt concentrate în câteva țări. **Decizia noastră: păstrăm toate regiunile, fără să completăm valorile lipsă**, și alegem pentru fiecare analiză doar coloanele de care are nevoie.

## 9. Perechile de regiuni
Rezolvăm dublura de conținut din secțiunea 4: păstrăm o singură apariție a fiecărei perechi, cea în care primul cod e mai mic alfabetic. Dispar și perechile unei regiuni cu ea însăși, pentru că un cod nu e mai mic decât el însuși.

**Anticipează:** câte perechi rămân din 298 de regiuni?

In [ ]:
perechi = sci[sci["regiune_1"] < sci["regiune_2"]]      # o singură ordine a fiecărei perechi
print(len(sci), "rânduri ->", len(perechi), "perechi")

298 × 297 / 2 = 44.253. Acum lipim indicatorii **de două ori**: o dată pentru capătul 1, o dată pentru capătul 2. `add_suffix("_1")` pune `_1` după fiecare nume de coloană, deci și `regiune` devine `regiune_1`, exact coloana după care lipim.

In [ ]:
atribute = regiuni.drop(columns=["tara", "denumire"])     # regiunea și cei șapte indicatori
atribute.add_suffix("_1").head(2)

In [ ]:
inainte = len(perechi)
perechi = perechi.merge(atribute.add_suffix("_1"), on="regiune_1", how="left")   # indicatorii capătului 1
perechi = perechi.merge(atribute.add_suffix("_2"), on="regiune_2", how="left")   # indicatorii capătului 2

print(inainte, "->", len(perechi), "perechi")      # regula integrării

## 10. Acoperirea migrației
Nu toate țările trimit la Eurostat migranții pe fiecare cetățenie. Numărăm, pentru fiecare țară de rezidență, câte cetățenii au o valoare.

In [ ]:
acoperire = migratie.groupby("tara_rezidenta")["persoane"].count()    # câte cetățenii cu valoare, pe țară
acoperire.sort_values().head(12)                                    # cele care raportează cel mai puțin

**Interpretează:** unele țări au 0: nu raportează deloc cetățeniile. Perechile cu ele ar avea migrația incompletă. Lipsa se scrie în raport, nu se ascunde.

Păstrăm ca **raportoare** țările care dau cel puțin 20 de cetățenii.

In [ ]:
raportoare = acoperire[acoperire >= 20].index
print(len(raportoare), "țări raportoare:", list(raportoare))

## 11. Perechile de țări
Un rând va fi o pereche de țări, cu SCI, migrația în **ambele sensuri** și zborurile. SCI pe țări e în `country.csv`, din C1; și aici apare Namibia.

In [ ]:
tari = pd.read_csv("data/country.csv", keep_default_na=False)
tari = tari[["user_country", "friend_country", "scaled_sci"]]
tari.columns = ["tara_1", "tara_2", "sci"]

tari = tari[tari["tara_1"] < tari["tara_2"]]                                      # o singură ordine
tari = tari[tari["tara_1"].isin(raportoare) & tari["tara_2"].isin(raportoare)]    # ambele țări raportează
print(len(tari), "perechi de țări")

Lipim migrația de două ori, ca indicatorii la regiuni. Întâi cetățenii țării 1 care trăiesc în țara 2: redenumim coloanele migrației ca să se potrivească cu cheile din `tari`. Apoi invers.

In [ ]:
inainte = len(tari)

din_1_in_2 = migratie.rename(columns={"cetatenie": "tara_1", "tara_rezidenta": "tara_2", "persoane": "din_1_in_2"})
tari = tari.merge(din_1_in_2, on=["tara_1", "tara_2"], how="left")      # cetățenii țării 1 din țara 2

din_2_in_1 = migratie.rename(columns={"cetatenie": "tara_2", "tara_rezidenta": "tara_1", "persoane": "din_2_in_1"})
tari = tari.merge(din_2_in_1, on=["tara_1", "tara_2"], how="left")      # cetățenii țării 2 din țara 1

print(inainte, "->", len(tari), "perechi")      # regula integrării

SCI nu are sens: legătura RO-IT e aceeași cu IT-RO. Îl comparăm deci cu migrația **în ambele sensuri, adunată**. Dacă unul dintre sensuri lipsește, suma rămâne lipsă: pandas propagă `NaN` în calcule, iar noi nu numim „mică” o migrație pe care nu o cunoaștem.

La zboruri, fiecare țară își raportează pasagerii, deci perechea apare de două ori (am văzut în C3). Păstrăm valoarea raportată de țara 1.

In [ ]:
tari["migranti"] = tari["din_1_in_2"] + tari["din_2_in_1"]      # o valoare lipsă în sumă dă lipsă

inainte = len(tari)
tari = tari.merge(zboruri, on=["tara_1", "tara_2"], how="left")   # pasagerii raportați de țara 1
print(inainte, "->", len(tari), "perechi")

**Exercițiu.** Perechile României, de la cei mai mulți migranți la cei mai puțini. `|` înseamnă „sau”: România poate fi oricare dintre cele două capete.

In [ ]:
# 1. Perechile în care tara_1 este "RO" sau tara_2 este "RO" (două condiții legate cu |)
# 2. Ordonează după migranti, de la cel mai mare, și afișează primele 10

## 12. Validarea finală
Înainte de salvare, verificăm regulile fiecărui tabel de analiză, într-o singură celulă: câte rânduri are, dacă cheia lui e unică și dacă variabila de explicat (SCI) e completă.

In [ ]:
print("regiuni:          ", len(regiuni), "rânduri;", regiuni.duplicated("regiune").sum(), "chei repetate")
print("perechi regiuni:  ", len(perechi), "rânduri;", perechi.duplicated(["regiune_1", "regiune_2"]).sum(), "chei repetate;",
      perechi["sci"].isna().sum(), "fără SCI")
print("perechi de țări:  ", len(tari), "rânduri;", tari.duplicated(["tara_1", "tara_2"]).sum(), "chei repetate;",
      tari["sci"].isna().sum(), "fără SCI")

Rezultatul așteptat: 298 de regiuni, 44.253 de perechi de regiuni și 276 de perechi de țări, 0 chei repetate peste tot și 0 perechi fără SCI. Dacă o cifră diferă, un pas de mai sus trebuie refăcut înainte de salvare.

**Anticipează pentru C5:** perechile cu mulți migranți sunt și cele cu SCI mare? Dar cele cu mulți pasageri?

## 13. Salvăm și scriem deciziile
Trei tabele de analiză, în `data/processed`:

In [ ]:
regiuni.to_csv("data/processed/regiuni_nuts2.csv", index=False)       # un rând pe regiune
perechi.to_csv("data/processed/perechi_regiuni.csv", index=False)     # un rând pe pereche de regiuni
tari.to_csv("data/processed/perechi_tari.csv", index=False)           # un rând pe pereche de țări

### Jurnalul deciziilor
Orice curățare înseamnă decizii. Le scriem, cu motivul lor, ca oricine să le poată verifica (în proiect, în `README.md`).

| Tema | Decizia noastră | De ce |
|---|---|---|
| Valori neobișnuite (populația 0 la Svalbard) | păstrate | sunt raportate așa de sursă |
| Codul `NA` (Namibia) | citit ca text | altfel 44 de rânduri devin lipsă |
| Regiuni fără date | păstrate, fără completare | lipsurile nu sunt întâmplătoare |
| Perechi duble A-B / B-A | o singură apariție | aceeași valoare, de două ori |
| Coduri EL/UK | aliniate la GR/GB | altfel pierdem țări la lipire |
| Țări care nu raportează migrația | scoase din perechile de țări | nu cunoaștem migrația lor |
| Zboruri raportate de ambele țări | valoarea raportată de țara 1 | o singură valoare pe pereche |

### Exercițiul echipei
1. Din ce țări sunt regiunile fără PIB?

In [ ]:
# 1. Regiunile în care pib_locuitor lipsește (isna)
# 2. Câte sunt, pe țară (value_counts pe coloana tara)

2. Găsiți perechea de țări a coridorului vostru în `tari`. Are migrația în ambele sensuri? Are zboruri? Dacă nu, de ce?

In [ ]:
# Perechea coridorului vostru: tara_1 și tara_2, în ordine alfabetică (de exemplu "IT" și "RO")

3. În depozitul echipei, pe `dev`: aplicați verificările din secțiunile 2-7 pe datele proprii, notați deciziile în `README.md` și urcați caietul (`git pull`, `git add`, `git commit`, `git push`).

## Ce ducem mai departe
În `data/processed` avem trei tabele de analiză: `regiuni_nuts2.csv`, `perechi_regiuni.csv` și `perechi_tari.csv`. În C5 le încărcăm într-o bază de date și le agregăm.

**Poți explica fără cod?** Ce arată `info()` și ce arată `value_counts()`? Prin ce diferă o cheie repetată de o dublură de conținut? De ce `NA` nu e întotdeauna o valoare lipsă? Ce face `how="left"`? De ce numărăm rândurile după fiecare `merge`? De ce o sumă cu o valoare lipsă rămâne lipsă? Ce verifică celula de validare finală?

### Surse
- [pandas: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas: Reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html)
- [pandas: Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html)
- [Eurostat, Glossary: Country codes](https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Glossary:Country_codes)